# ML Learning Coach (English): Part 2, QLoRA training
Loads `Qwen/Qwen2.5-1.5B-Instruct` in 4-bit, adds a LoRA adapter and trains it on `train.jsonl`.
Outputs: **our own adapter**, a loss curve and `training_info.json` (proof of a real training run).
Runtime > Change runtime type > **T4 GPU**. Run the cells from top to bottom, and make sure the LAST cell finishes.

### 1) GPU and libraries

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets

### 2) Connect Drive
The adapter is written to Drive so it survives a closed session.

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/ml-coach-en"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(os.listdir(DRIVE_DIR))

### 3) Get the training file

In [ ]:
import shutil
from google.colab import files

def fetch(name):
    if os.path.exists(name):
        return
    p = f"{DRIVE_DIR}/{name}"
    if os.path.exists(p):
        shutil.copy(p, name)
        return
    print(f"{name} not found, please upload it:")
    up = files.upload()
    k = list(up)[0]
    if k != name:
        shutil.move(k, name)
    shutil.copy(name, p)

fetch("train.jsonl")
print("train.jsonl ready:", os.path.getsize("train.jsonl") // 1024, "KB")

### 4) Read the data and split off a validation set
12 examples are kept aside. The model never trains on them, so we can watch for overfitting.

In [ ]:
import json, random, hashlib
from datasets import Dataset

raw = open("train.jsonl", "rb").read()
DATA_SHA = hashlib.sha256(raw).hexdigest()[:12]
rows = [json.loads(l) for l in raw.decode("utf-8").splitlines() if l.strip()]

random.seed(42)
random.shuffle(rows)
VAL_N = 12
val_rows, train_rows = rows[:VAL_N], rows[VAL_N:]
print(f"Total: {len(rows)} | train: {len(train_rows)} | validation: {len(val_rows)} | dataset SHA: {DATA_SHA}")

### 5) Tokenization and masking
We want the model to learn **only from the coach answers**. System prompt and student messages get `labels = -100`, so no loss is computed on them.
In multi-turn conversations, every coach answer is learned.

In [ ]:
from transformers import AutoTokenizer

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MARKER, END = "<|im_start|>assistant\n", "<|im_end|>"
MAX_LEN = 2048


def encode(messages):
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    ids, labels, pos = [], [], 0
    while True:
        s = text.find(MARKER, pos)
        if s == -1:                                   # trailing part (masked)
            piece = tokenizer(text[pos:], add_special_tokens=False)["input_ids"]
            ids += piece; labels += [-100] * len(piece)
            break
        prompt = tokenizer(text[pos:s + len(MARKER)], add_special_tokens=False)["input_ids"]
        ids += prompt; labels += [-100] * len(prompt)  # system/user (masked)
        e = text.find(END, s + len(MARKER))
        answer = tokenizer(text[s + len(MARKER):e + len(END)], add_special_tokens=False)["input_ids"]
        ids += answer; labels += answer                # coach answer (learned)
        pos = e + len(END)
    return {"input_ids": ids, "labels": labels, "attention_mask": [1] * len(ids)}


def build(rs):
    ds = Dataset.from_list([encode(r["messages"]) for r in rs])
    return ds.filter(lambda x: len(x["input_ids"]) <= MAX_LEN)


train_ds, val_ds = build(train_rows), build(val_rows)
lens = [len(x) for x in train_ds["input_ids"]]
print(f"Train: {len(train_ds)} | validation: {len(val_ds)} | longest: {max(lens)} tokens | mean: {sum(lens)//len(lens)} tokens")

# Masking check: shows what the model actually LEARNS from
ex = train_ds[0]
learned = [t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100]
print("\n--- What the model learns from (should be coach answers only) ---")
print(tokenizer.decode(learned))

### 6) Load the model in 4-bit and add the LoRA adapter
- **4-bit (NF4):** the base weights stay frozen and compressed, which saves memory.
- **LoRA (r=16):** only small adapter matrices are trained, about 1% of the model.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)   # also enables gradient checkpointing

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

### 7) Training
3 epochs, effective batch size 8 (2 x 4 accumulation), learning rate 2e-4, cosine schedule. Validation loss is measured after every epoch.

In [ ]:
import inspect, time
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, set_seed

set_seed(42)
params = inspect.signature(TrainingArguments.__init__).parameters
eval_kw = "eval_strategy" if "eval_strategy" in params else "evaluation_strategy"

args = TrainingArguments(
    output_dir="/content/checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    optim="paged_adamw_8bit",
    logging_steps=2,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
    seed=42,
    **{eval_kw: "epoch"},
)
collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=val_ds, data_collator=collator)

t0 = time.time()
trainer.train()
TRAIN_SECONDS = round(time.time() - t0)
print(f"\nTraining finished in {TRAIN_SECONDS} seconds")

### 8) Loss curve

In [ ]:
import matplotlib.pyplot as plt

log = trainer.state.log_history
train_pts = [(l["step"], l["loss"]) for l in log if "loss" in l]
eval_pts = [(l["step"], l["eval_loss"]) for l in log if "eval_loss" in l]

plt.figure(figsize=(7, 4))
plt.plot(*zip(*train_pts), label="train loss")
if eval_pts:
    plt.plot(*zip(*eval_pts), "o-", label="validation loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3)
plt.title("QLoRA training: loss")
plt.savefig("loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print("First train loss:", round(train_pts[0][1], 3))
print("Last train loss: ", round(train_pts[-1][1], 3))
for s, v in eval_pts:
    print(f"validation loss (step {s}):", round(v, 3))

### 9) Save the adapter and the training evidence (IMPORTANT: this cell must finish)
- `adapter/`: our trained LoRA weights (on Drive, plus `adapter.zip` for your computer)
- `training_info.json`: date, hyperparameters, dataset SHA, losses. It proves the run is ours.
- `training_log.json` and `loss_curve.png`: evidence for the README.

In [ ]:
import datetime
import transformers, peft

ADAPTER_DIR = f"{DRIVE_DIR}/adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

info = {
    "finished_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "base_model": BASE,
    "gpu": torch.cuda.get_device_name(0),
    "dataset_sha256_prefix": DATA_SHA,
    "n_train": len(train_ds), "n_val": len(val_ds),
    "lora": {"r": 16, "alpha": 32, "dropout": 0.05, "target_modules": sorted(lora.target_modules)},
    "hyperparameters": {"epochs": 3, "batch_size": 2, "grad_accum": 4, "learning_rate": 2e-4,
                        "scheduler": "cosine", "warmup_ratio": 0.05, "optimizer": "paged_adamw_8bit",
                        "fp16": True, "max_len": MAX_LEN, "seed": 42},
    "train_seconds": TRAIN_SECONDS,
    "first_train_loss": train_pts[0][1], "last_train_loss": train_pts[-1][1],
    "eval_losses": eval_pts,
    "versions": {"transformers": transformers.__version__, "peft": peft.__version__, "torch": torch.__version__},
}
json.dump(info, open("training_info.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
json.dump(log, open("training_log.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

for name in ["training_info.json", "training_log.json", "loss_curve.png"]:
    shutil.copy(name, f"{DRIVE_DIR}/{name}")

shutil.make_archive("/content/adapter", "zip", ADAPTER_DIR)
print("Adapter files:", os.listdir(ADAPTER_DIR))
print("Written to Drive:", ADAPTER_DIR)

for name in ["/content/adapter.zip", "training_info.json", "training_log.json", "loss_curve.png"]:
    files.download(name)